# 🌿 Mint Leaf AI — STEP 7: Reusable Multi-Model Training Framework Validation

Welcome to **Step 7** of the Mint Leaf AI project. In this notebook (`06_model_framework_validation.ipynb`), we build and validate our reusable, multi-model PyTorch training and evaluation framework.

--- 

### 🔬 Framework Design & Validation Purpose:
- **Purpose**: This run uses **ONE representative model (`resnet18`, 3 epochs)** to validate the complete end-to-end training pipeline.
- **Pipeline Validation Order**:
  ```text
  data/processed/ ➔ DataLoader ➔ ResNet18 Model ➔ Trainer (GPU/AMP) ➔ Validation ➔ Checkpoint (.pt) ➔ Test Evaluation ➔ 6-Class Metrics ➔ Visualization
  ```
- **Strict Verification**: Prove that GPU hardware is detected, batch shapes are valid, label mappings are correct, loss decreases, checkpoint files are written to disk, and confusion matrix reports are generated without errors.

--- 

⚠️ **Constraint Checklist**:
- [x] Framework fully created under `models/`, `training/`, and `evaluation/`.
- [x] ONLY 1 representative model trained for pipeline validation.
- [x] Do NOT train remaining 24 models yet.
- [x] STOP after single-model validation and wait for user approval.

## 🛠️ Section 1: Environment, Hardware GPU Detection & Imports

In [1]:
import os
import sys
import json
import time
from pathlib import Path

import torch
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

# Formatting
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.autolayout"] = True

# Environment Detection
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running in Google Colab ML Laboratory.")
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/mint-leaf-ai')
else:
    print("💻 Running in Local Antigravity IDE Environment.")
    cwd = Path(os.getcwd()).resolve()
    BASE_PATH = cwd.parent if cwd.name == 'notebooks' else cwd

sys.path.append(str(BASE_PATH))

# Hardware GPU Detection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Hardware Accelerator Device: {device}")
if device.type == 'cuda':
    print(f"   GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Available:  {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("   CPU Mode Active (PyTorch Multi-threading)")

# Framework Imports
from training.data.dataset import get_dataloaders
from models.architectures.factory import build_model, get_model_metrics
from training.trainers.trainer import PyTorchTrainer
from evaluation.metrics.evaluator import ModelEvaluator
from evaluation.visualization.plotter import plot_confusion_matrix, plot_training_history

## 📦 Section 2: DataLoaders & Batch Dimension Audit

In [2]:
config_path = BASE_PATH / 'models' / 'configs' / 'resnet18_baseline.json'
with open(config_path, 'r') as f:
    config = json.load(f)

print(f"📋 Loaded Model Configuration: {config_path.name}")
print(json.dumps(config, indent=4))

# Construct DataLoaders
processed_dir = BASE_PATH / 'data' / 'processed'
loaders = get_dataloaders(
    processed_dir=processed_dir,
    batch_size=config.get("batch_size", 32),
    img_size=config.get("input_resolution", 224),
    num_workers=0
)

train_loader = loaders["train"]
val_loader = loaders["val"]
test_loader = loaders["test"]
classes = loaders["classes"]

print(f"\n📊 DataLoaders Construction Summary:")
print(f"- Train Batches:      {len(train_loader)} ({len(train_loader.dataset)} images)")
print(f"- Validation Batches: {len(val_loader)} ({len(val_loader.dataset)} images)")
print(f"- Test Batches:       {len(test_loader)} ({len(test_loader.dataset)} images)")
print(f"- Label Classes (6):  {classes}")

# Inspect Single Batch Dimensions
sample_images, sample_labels, _ = next(iter(train_loader))
print(f"\n🔍 Batch Dimension Check:")
print(f"  Images Tensor Shape: {sample_images.shape} (Batch, Channel, Height, Width)")
print(f"  Labels Tensor Shape: {sample_labels.shape}")
print(f"  Unique Labels in Batch: {sample_labels.unique().tolist()}")

## 🏗️ Section 3: Model Architecture & Parameter Inspection

In [3]:
# Build Model
model = build_model(model_name=config["model_name"], num_classes=config["num_classes"], pretrained=config["pretrained"])
model_meta = get_model_metrics(model, device=device.type)

print(f"🏗️ Architecture Summary for '{config['model_name']}':")
print(f"- Total Parameters:     {model_meta['total_params']:,}")
print(f"- Trainable Parameters: {model_meta['trainable_params']:,}")
print(f"- Estimated Size:       {model_meta['model_size_mb']} MB")

## 🚀 Section 4: Trainer Fit Execution (ResNet18 Representative Model)

In [4]:
# Update config paths relative to BASE_PATH
config["checkpoint_path"] = str(BASE_PATH / config["checkpoint_path"])
config["history_path"] = str(BASE_PATH / config["history_path"])

# Class Counts for Optional Loss Weighting
class_counts = {cls: len(list((processed_dir / 'train' / cls).glob('*.jpg'))) for cls in classes}

# Instantiate Trainer Engine
trainer = PyTorchTrainer(config=config, class_counts=class_counts)

# Fit Model for 3 Validation Epochs
history = trainer.fit(train_loader, val_loader)
print("\n✅ Training Loop Complete!")

## 📈 Section 5: Checkpoint & Physical Artifact Verification

In [5]:
ckpt_path = Path(config["checkpoint_path"])
hist_path = Path(config["history_path"])

print("🔍 Physical File Verification:")
print(f"- Checkpoint File ({ckpt_path.name}): {ckpt_path.exists()} (Size: {ckpt_path.stat().st_size / (1024*1024):.2f} MB)")
print(f"- History File ({hist_path.name}):    {hist_path.exists()}")

assert ckpt_path.exists(), "Checkpoint file missing!"
assert hist_path.exists(), "History file missing!"
print("✅ Checkpoint and history files physically confirmed on disk!")

## 🧪 Section 6: Comprehensive Test Evaluation & Metrics Report

In [6]:
# Load Best Model Weights from Checkpoint
best_checkpoint = torch.load(ckpt_path, map_location=device)
trainer.model.load_state_dict(best_checkpoint['model_state_dict'])

# Evaluator Engine
evaluator = ModelEvaluator(trainer.model, classes=classes, device=device.type)
eval_results = evaluator.evaluate(test_loader, checkpoint_path=ckpt_path)

summary = eval_results["summary"]
per_class_df = eval_results["per_class_df"]
cm_df = eval_results["confusion_matrix_df"]

print("\n📊 Test Evaluation Summary:")
for k, v in summary.items():
    print(f"  - {k}: {v}")

print("\n📋 Per-Class Performance Breakdown:")
display(per_class_df)

print("\n🔢 Confusion Matrix:")
display(cm_df)

## 🎨 Section 7: Visualization Dashboards & Report Export

In [7]:
# 1. Plot Confusion Matrix
cm_save_path = BASE_PATH / 'outputs' / 'visualizations' / 'resnet18_confusion_matrix.png'
plot_confusion_matrix(cm_df, save_path=cm_save_path, title="ResNet18 Baseline Confusion Matrix (6 Classes)")

# 2. Plot Training History Curves
hist_save_path = BASE_PATH / 'outputs' / 'visualizations' / 'resnet18_training_history.png'
plot_training_history(history, save_path=hist_save_path, title="ResNet18 Validation Run Metrics")

# 3. Save Final Framework Evaluation JSON & Markdown
framework_report_dir = BASE_PATH / 'outputs' / 'reports' / 'model_framework'
framework_report_dir.mkdir(parents=True, exist_ok=True)

report_json_path = framework_report_dir / 'single_model_framework_validation_report.json'
with open(report_json_path, 'w', encoding='utf-8') as f:
    json.dump({
        'config': config,
        'summary_metrics': summary,
        'per_class_performance': per_class_df.to_dict(orient='records'),
        'confusion_matrix': cm_df.to_dict()
    }, f, indent=4)

print(f"\n💾 Framework Validation JSON Report exported to: {report_json_path}")